In [1]:
import os
import sys
import pdb
import six
import random
import lmdb
from PIL import Image
import numpy as np
import math
from collections import OrderedDict
from itertools import chain
import logging


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import sampler
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.nn.utils.clip_grad import clip_grad_norm_
from torch.utils.data import random_split
sys.path.insert(0, '../')
from src.utils.utils import AverageMeter, Eval, OCRLabelConverter
from src.utils.utils import EarlyStopping, gmkdir
from src.optim.optimizer import STLR
from src.utils.utils import gaussian
from tqdm import *


In [ ]:
%%capture

### Data generation using trdg


#!/usr/bin/env python3
import os
from trdg.generators import GeneratorFromStrings

def main():
    # Input file containing words (one word per line)
    input_file = '/home/akash/ws/limited_supervision_ocr/Adapting-OCR/val_words.txt'
    # Directory where images will be saved
    output_dir = 'val_diner_Fatt_output_images'
    # File to write the image paths and corresponding labels
    results_file = 'val_diner_fatt_val_results.txt'
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Read words from the input text file
    with open(input_file, 'r', encoding='utf-8') as f:
        words = [line.strip() for line in f if line.strip()]
    
    # Create the generator: count=1 means one image per word
    generator = GeneratorFromStrings(words, count=44000, fonts=["/home/akash/ws/limited_supervision_ocr/Adapting-OCR/fonts/Diner-Fatt.ttf"])
    
    # Open the results file for writing output info
    with open(results_file, 'w', encoding='utf-8') as rf:
        image_index = 0
        for img, label in generator:
            image_index += 1
            # Create a filename for the image (e.g., img_0001.png)
            filename = f"img_{image_index:04d}.png"
            image_path = os.path.join(os.getcwd(), output_dir, filename)
            # Save the generated image
            img.save(image_path)
            # Write the image path and label (separated by a tab) to the results file
            rf.write(f"{image_path}\t{label}\n")
            print(f"Saved {image_path} with label: {label}")

if __name__ == "__main__":
    main()


In [ ]:
%%capture
##Preprocessing --> making pickle file

import os
import pickle
import numpy as np
from PIL import Image

def create_pickle_dataset_from_txt(txt_file, output_path, base_dir='', split='train', delimiter=' '):
    """
    Creates a pickle file from a text file containing file paths and labels
    
    Args:
        txt_file: Path to text file with file paths and labels
        output_path: Path to save the output pickle file
        base_dir: Base directory to prepend to file paths (optional)
        split: Dataset split name (default 'train')
        delimiter: Separator between file path and label (default ' ')
    """
    dataset = {split: []}
    
    with open(txt_file, 'r') as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # Split into file path and label
        parts = line.split(delimiter)
        if len(parts) < 2:
            print(f"Skipping invalid line: {line}")
            continue
            
        rel_path, label = parts[0], delimiter.join(parts[1:])
        full_path = os.path.join(base_dir, rel_path)
        
        if not os.path.exists(full_path):
            print(f"File not found: {full_path}")
            continue
            
        try:
            img = Image.open(full_path)
            img_array = np.array(img)
            dataset[split].append((img_array, label))
        except Exception as e:
            print(f"Error processing {full_path}: {str(e)}")
    
    with open(output_path, 'wb') as f:
        pickle.dump(dataset, f)

# Example usage
if __name__ == '__main__':
    create_pickle_dataset_from_txt(
        txt_file='/home/akash/ws/limited_supervision_ocr/Adapting-OCR/notebooks/val_diner_fatt_val_results.txt',
        output_path='/home/akash/ws/limited_supervision_ocr/Adapting-OCR/notebooks/val_diner_Fatt_output_images/English.data.pkl',
        base_dir='/home/akash/ws/limited_supervision_ocr/Adapting-OCR/notebooks/output_images',
        split='val',
        delimiter='\t'  # Could also use '\t' for tab-separated
    )

    